<a href="https://colab.research.google.com/github/akshita123454/mineral-exploration-cps/blob/main/elevation_and_slope_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install earthengine-api geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 27.4 MB/s eta 0:00:00


In [7]:
import ee
ee.Authenticate()

In [8]:
ee.Initialize(project='mineral-exploration-project')

In [9]:
print(ee.Image('USGS/SRTMGL1_003').getInfo())

{'type': 'Image', 'bands': [{'id': 'elevation', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': -32768, 'max': 32767}, 'dimensions': [1296001, 417601], 'crs': 'EPSG:4326', 'crs_transform': [0.0002777777777777778, 0, -180.0001388888889, 0, -0.0002777777777777778, 60.00013888888889]}], 'version': 1641990767055141, 'id': 'USGS/SRTMGL1_003', 'properties': {'system:visualization_0_min': '0.0', 'type_name': 'Image', 'keywords': ['dem', 'elevation', 'geophysical', 'nasa', 'srtm', 'topography', 'usgs'], 'thumb': 'https://mw1.google.com/ges/dd/images/SRTM90_V4_thumb.png', 'description': '<p>The Shuttle Radar Topography Mission (SRTM, see <a href="https://onlinelibrary.wiley.com/doi/10.1029/2005RG000183/full">Farr\net al. 2007</a>)\ndigital elevation data is an international research effort that\nobtained digital elevation models on a near-global scale. This\nSRTM V3 product (SRTM Plus) is provided by NASA JPL\nat a resolution of 1 arc-second (approximately 30m).</p><p>This dataset

In [10]:
from google.colab import files
uploaded = files.upload()

Saving mineral_data.csv to mineral_data.csv


In [32]:
import pandas as pd
df = pd.read_csv('mineral_data.csv')

/tmp/ipykernel_1602/314531497.py:2: DtypeWarning: Columns (13,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('mineral_data.csv')


In [13]:
dem = ee.Image('USGS/SRTMGL1_003')

In [14]:
def get_elevation(lat, lon):
    point = ee.Geometry.Point([lon, lat])
    elevation = dem.sample(point, 30).first().get('elevation')
    return elevation.getInfo()

In [15]:
def df_to_ee(df):
    features = []
    for i in range(len(df)):
        lat = df.iloc[i]['latitude']
        lon = df.iloc[i]['longitude']

        point = ee.Geometry.Point([lon, lat])
        feature = ee.Feature(point, {
            'Latitude': lat,
            'Longitude': lon
        })

        features.append(feature)

    return ee.FeatureCollection(features)

fc = df_to_ee(df)

In [16]:
elevation_fc = dem.sampleRegions(
    collection=fc,
    scale=30,
    geometries=True
)

In [17]:
import pandas as pd

def ee_to_df(fc):
    features = fc.getInfo()['features']

    data = []
    for f in features:
        props = f['properties']
        data.append(props)

    return pd.DataFrame(data)



In [18]:
dem = ee.Image('USGS/SRTMGL1_003')
slope = ee.Terrain.slope(dem)

combined = dem.addBands(slope)

In [19]:
sampled = combined.sampleRegions(
    collection=fc,
    scale=30,
    geometries=True
)

In [20]:

# Remove rows with missing coordinates
df = df.dropna(subset=['latitude', 'longitude'])

# Convert to numeric (in case of string issues)
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

# Remove again after conversion
df = df.dropna(subset=['latitude', 'longitude'])

In [21]:
df = df[
    (df['latitude'] >= -90) & (df['latitude'] <= 90) &
    (df['longitude'] >= -180) & (df['longitude'] <= 180)
]

In [22]:
df = df.reset_index(drop=True)

In [23]:
batch_size = 2000   # smaller = safer

for i in range(0, len(df), batch_size):
    print(f"Processing batch {i} to {i+batch_size}")

    batch = df.iloc[i:i+batch_size]

    # Convert batch only
    fc = df_to_ee(batch)

    dem = ee.Image('USGS/SRTMGL1_003')
    slope = ee.Terrain.slope(dem)
    combined = dem.addBands(slope)

    sampled = combined.sampleRegions(
        collection=fc,
        scale=30
    )

    task = ee.batch.Export.table.toDrive(
        collection=sampled,
        description=f'batch_{i}',
        fileFormat='CSV'
    )

    task.start()

Processing batch 0 to 2000
Processing batch 2000 to 4000
Processing batch 4000 to 6000
Processing batch 6000 to 8000
Processing batch 8000 to 10000
Processing batch 10000 to 12000
Processing batch 12000 to 14000
Processing batch 14000 to 16000
Processing batch 16000 to 18000
Processing batch 18000 to 20000
Processing batch 20000 to 22000
Processing batch 22000 to 24000
Processing batch 24000 to 26000
Processing batch 26000 to 28000
Processing batch 28000 to 30000
Processing batch 30000 to 32000
Processing batch 32000 to 34000
Processing batch 34000 to 36000
Processing batch 36000 to 38000
Processing batch 38000 to 40000
Processing batch 40000 to 42000
Processing batch 42000 to 44000
Processing batch 44000 to 46000
Processing batch 46000 to 48000
Processing batch 48000 to 50000
Processing batch 50000 to 52000
Processing batch 52000 to 54000
Processing batch 54000 to 56000
Processing batch 56000 to 58000
Processing batch 58000 to 60000
Processing batch 60000 to 62000
Processing batch 620

In [24]:
import pandas as pd
import glob

files = glob.glob("/content/*.csv")

df_list = [pd.read_csv(f) for f in files]

final_df = pd.concat(df_list, ignore_index=True)

/tmp/ipykernel_1602/1748044921.py:6: DtypeWarning: Columns (13,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list = [pd.read_csv(f) for f in files]


In [25]:
from google.colab import files

uploaded = files.upload()

Saving batch_36000.csv to batch_36000.csv
Saving batch_56000.csv to batch_56000.csv
Saving batch_86000.csv to batch_86000.csv
Saving batch_120000.csv to batch_120000.csv
Saving batch_276000.csv to batch_276000.csv
Saving batch_6000.csv to batch_6000.csv
Saving batch_16000.csv to batch_16000.csv
Saving batch_26000.csv to batch_26000.csv
Saving batch_30000.csv to batch_30000.csv
Saving batch_38000.csv to batch_38000.csv
Saving batch_42000.csv to batch_42000.csv
Saving batch_46000.csv to batch_46000.csv
Saving batch_50000.csv to batch_50000.csv
Saving batch_70000.csv to batch_70000.csv
Saving batch_76000.csv to batch_76000.csv
Saving batch_78000.csv to batch_78000.csv
Saving batch_80000.csv to batch_80000.csv
Saving batch_92000.csv to batch_92000.csv
Saving batch_94000.csv to batch_94000.csv
Saving batch_98000.csv to batch_98000.csv
Saving batch_102000.csv to batch_102000.csv
Saving batch_104000.csv to batch_104000.csv
Saving batch_106000.csv to batch_106000.csv
Saving batch_110000.csv to

In [26]:
import pandas as pd

df_list = [pd.read_csv(f, low_memory=False) for f in files]

gee_df = pd.concat(df_list, ignore_index=True)

print("Merged shape:", gee_df.shape)
gee_df.head()

TypeError: 'module' object is not iterable

In [27]:
import glob

files = glob.glob('batch_*.csv')

print("Total files found:", len(files))
print(files[:5])  # show first few


Total files found: 153
['batch_224000.csv', 'batch_14000.csv', 'batch_60000.csv', 'batch_260000.csv', 'batch_158000.csv']


In [28]:
import pandas as pd

df_list = []

for f in files:
    df = pd.read_csv(f, low_memory=False)
    df_list.append(df)

gee_df = pd.concat(df_list, ignore_index=True)

print("Merged shape:", gee_df.shape)
gee_df.head()

Merged shape: (294312, 6)


,system:index,Latitude,Longitude,elevation,slope,.geo
0,0_0,46.14983,-120.02053,326,6.464861,"{""type"":""MultiPoint"",""coordinates"":[]}"
1,1_0,44.75750,-88.20144,245,1.601543,"{""type"":""MultiPoint"",""coordinates"":[]}"
2,2_0,42.96141,-89.13147,304,1.854334,"{""type"":""MultiPoint"",""coordinates"":[]}"
3,3_0,44.14610,-91.69736,336,5.238097,"{""type"":""MultiPoint"",""coordinates"":[]}"
4,4_0,44.81670,-89.12397,351,5.531942,"{""type"":""MultiPoint"",""coordinates"":[]}"


In [29]:
gee_df.rename(columns={
    'Latitude': 'latitude',
    'Longitude': 'longitude'
}, inplace=True)

In [30]:
gee_df.columns

Index(['system:index', 'latitude', 'longitude', 'elevation', 'slope', '.geo'], dtype='object')

In [33]:
print("Original DF columns:")
print(df.columns)

print("\nGEE DF columns:")
print(gee_df.columns)

Original DF columns:
Index(['site_name', 'latitude', 'longitude', 'region', 'country', 'state',
       'county', 'com_type', 'commod1', 'commod2', 'commod3', 'oper_type',
       'dep_type', 'prod_size', 'dev_stat', 'ore', 'gangue', 'work_type',
       'names', 'ore_ctrl', 'hrock_type', 'arock_type'],
      dtype='object')

GEE DF columns:
Index(['system:index', 'latitude', 'longitude', 'elevation', 'slope', '.geo'], dtype='object')


In [34]:
df.rename(columns={
    'Latitude': 'latitude',
    'Longitude': 'longitude'
}, inplace=True)

In [35]:
merged_df = df.merge(
    gee_df[['latitude','longitude','elevation','slope']],
    on=['latitude','longitude'],
    how='left'
)

print("Merged shape:", merged_df.shape)
merged_df.head()

Merged shape: (440968, 24)


,site_name,latitude,longitude,region,country,state,county,com_type,commod1,commod2,...,dev_stat,ore,gangue,work_type,names,ore_ctrl,hrock_type,arock_type,elevation,slope
0,Lookout Prospect,55.05612,-132.14344,NaN,United States,Alaska,NaN,M,Copper,"Gold, Silver",...,Occurrence,"Chalcopyrite, Covellite, Pyrite","Quartz, Sericite",NaN,"Conundrum, Mammoth, Wakefield Minerals Co.",NaN,Schist,NaN,417.0,34.061240
1,Lucky Find Prospect,55.52751,-132.68514,NaN,United States,Alaska,NaN,M,Copper,Gold,...,Occurrence,"Chalcopyrite, Pyrite","Calcite, Quartz, Siderite",Underground,NaN,Vein Follows Contact,Diabase,NaN,773.0,29.553010
2,Mccullough Prospect,55.97751,-132.99906,NaN,United States,Alaska,NaN,M,Copper,NaN,...,Occurrence,"Chalcopyrite, Pyrite, Sphalerite",Quartz,NaN,"Claims: Horseshoe, Copper, Lake Bay",NaN,Siltstone,NaN,58.0,3.311626
3,Lucky Jim Claim,55.52195,-132.68653,NaN,United States,Alaska,NaN,M,Gold,NaN,...,Occurrence,"Galena, Malachite, Pyrite",NaN,NaN,NaN,NaN,Granite,Granite,653.0,20.563173
4,Matilda Occurrence,55.14556,-132.05233,NaN,United States,Alaska,NaN,M,Gold,NaN,...,Occurrence,Pyrite,NaN,NaN,NaN,NaN,Mica Schist,NaN,36.0,8.804730


In [36]:
merged_df.columns

Index(['site_name', 'latitude', 'longitude', 'region', 'country', 'state',
       'county', 'com_type', 'commod1', 'commod2', 'commod3', 'oper_type',
       'dep_type', 'prod_size', 'dev_stat', 'ore', 'gangue', 'work_type',
       'names', 'ore_ctrl', 'hrock_type', 'arock_type', 'elevation', 'slope'],
      dtype='object')

In [37]:
merged_df.to_csv('elev_and_slope.csv', index=False)

In [39]:
from google.colab import files
files.download('elev_and_slope.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>